# 📚 SFT Distillation — Teaching Smaller Models
## Microsoft Build 2026 | LAB231 — Phase 2

---

### The Idea

> *"We have a smart but expensive teacher (GPT-5.4). Can we train cheaper students to match it — or even beat it — on this specific task?"*

**Supervised Fine-Tuning (SFT)** distillation:
1. Run the teacher on hundreds of scenarios → collect its correct outputs
2. Use those outputs as training data for smaller models
3. The students learn the *specific patterns* — which tools, which order, which calculations
4. Result: **specialized student > general teacher** (at lower cost!)

### What you'll see:

| # | Section | What we do |
|---|---------|------------|
| 1 | Training Data | Explore the teacher traces that become training examples |
| 2 | Submit SFT Jobs | One API call per model — Azure handles the rest |
| 3 | Results | Students beat the teacher at a fraction of the cost |
| 4 | Deploy & Test | Swap the model, same agent code |

> 💡 **Speaker notes:** [`docs/2_sft_distillation.md`](../docs/2_sft_distillation.md)

---

## 1. Training Data — The Teacher's Traces

The training data comes directly from **GPT-5.4's production traces**:

```
Teacher (GPT-5.4)                          Students
      │                                      │
      │  481 correct traces                  │
      │  (user message + full tool           │
      │   call sequence + resolution)        │
      │                                      │
      └──────────────────────────────────────┘
                     │         │         │
                     ▼         ▼         ▼
                 GPT-4.1   4.1-mini  4.1-nano
```

### Key files:

| File | Description |
|------|-------------|
| [`data/retail_seed.jsonl`](../data/retail_seed.jsonl) | 481 teacher traces — the raw training data |
| [`data/retail_train.jsonl`](../data/retail_train.jsonl) | Formatted training set |
| [`data/retail_val.jsonl`](../data/retail_val.jsonl) | Validation set (held out) |

### Format:
Each example is a standard **chat completion** with tool calls embedded — the student learns the entire conversation flow, not just the final answer.

> 🔑 *"The training data IS the teacher's traces. Same format as production. No manual labeling needed."*

In [ ]:
# Peek at the training data structure
import json

with open("../data/retail_seed.jsonl") as f:
    examples = [json.loads(line) for line in f]

print(f"📚 Training examples: {len(examples)}")
print(f"\n--- Example scenario ---")
ex = examples[0]
messages = ex.get("messages", [])
print(f"User: {messages[0]['content'][:100]}...")
print(f"\nConversation turns: {len(messages)}")

# Count tool calls in the conversation
tool_calls = [m for m in messages if m.get("role") == "assistant" and m.get("tool_calls")]
print(f"Tool call turns: {len(tool_calls)}")
if tool_calls:
    for tc_msg in tool_calls:
        for tc in tc_msg.get("tool_calls", []):
            print(f"  → {tc['function']['name']}")

---

## 2. Submit Fine-Tuning Jobs

SFT with Azure AI Foundry is **one API call**. Azure handles:
- Compute provisioning
- Checkpointing & early stopping
- Validation loss tracking
- Model registration & deployment

We fine-tune **3 students** from the same teacher data:

| Student Model | Why |
|--------------|-----|
| GPT-4.1 | Full-size — maximum learning capacity |
| GPT-4.1-mini | Balanced — good quality at low cost |
| GPT-4.1-nano | Smallest — can it learn this at all? |

> ⏱️ Training takes ~1-2 hours per model. Jobs are pre-completed for this demo.

In [ ]:
import os
import requests
from datetime import datetime
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
parts = PROJECT_ENDPOINT.split("/")
ACCOUNT = parts[2].split(".")[0]
PROJECT = parts[-1]
BASE_URL = f"{PROJECT_ENDPOINT}/openai"

credential = DefaultAzureCredential()

def get_headers():
    token = credential.get_token("https://ai.azure.com/.default").token
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print(f"✅ Connected to: {ACCOUNT}/{PROJECT}")

In [ ]:
# Upload training & validation data
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

train_dataset = client.datasets.upload_file(
    file_path="../data/retail_train.jsonl",
    name=f"retail-sft-train-{TIMESTAMP}",
    version="1.0"
)
print(f"✅ Training data uploaded: {train_dataset.name} ({train_dataset.id})")

val_dataset = client.datasets.upload_file(
    file_path="../data/retail_val.jsonl",
    name=f"retail-sft-val-{TIMESTAMP}",
    version="1.0"
)
print(f"✅ Validation data uploaded: {val_dataset.name} ({val_dataset.id})")

In [ ]:
# Submit SFT fine-tuning jobs
# The API call is beautifully simple:

MODELS_TO_FINETUNE = ["gpt-4.1", "gpt-4.1-mini", "gpt-4.1-nano"]

jobs = {}
for model in MODELS_TO_FINETUNE:
    job_body = {
        "model": model,
        "training_file": train_dataset.id,
        "validation_file": val_dataset.id,
        "suffix": "retail-sft",
        "method": {"type": "supervised_fine_tuning"}
    }
    url = f"{BASE_URL}/fine_tuning/jobs?api-version=2025-11-15-preview"
    resp = requests.post(url, headers=get_headers(), json=job_body)
    if resp.status_code in [200, 201]:
        job_data = resp.json()
        jobs[model] = job_data["id"]
        print(f"  🚀 {model:<14} → job {job_data['id'][:12]}... (status: {job_data['status']})")
    else:
        print(f"  ❌ {model}: {resp.status_code} - {resp.text[:150]}")

print(f"\n✅ {len(jobs)} SFT jobs submitted")
print(f"⏱️  Training takes ~1-2 hours. Check Azure AI Foundry → Fine-tuning for progress.")

---

## 3. Results — Students Beat the Teacher! 🎉

After SFT training completes, the fine-tuned models are deployed as:

| Student | Deployment Name |
|---------|----------------|
| GPT-4.1 fine-tuned | `gpt-4-1-2025-04-14-ft-626f359e91b848f18189e24b3e8e39b9` |
| GPT-4.1-mini fine-tuned | `gpt-4.1-mini-2025-04-14.ft-2eb6d31ecccd4796bce8fd9ccdc1529d` |
| GPT-4.1-nano fine-tuned | `gpt-4-1-nano-2025-04-14-ft-ff0be97034bf49359c038f622e395d88` |

### Before vs After:

| Model | Base (no FT) | After SFT | Improvement |
|-------|:---:|:---:|:---:|
| GPT-4.1-mini | 45.2% | **71.0%** ⬆️ | +25.8 pp |
| GPT-4.1-nano | 1.6% | **37.1%** ⬆️ | +35.5 pp |
| GPT-5.4 (teacher) | 64.5% | — | baseline |

### 🔑 The Punchline:

> *"The student **outperforms the teacher**! GPT-4.1-mini SFT at 71% beats GPT-5.4 at 64.5%.  
> Why? Because the student is **specialized** — it only needs to be good at THIS task.  
> And it costs **3-5x less** per request."*

### But we can do better...

SFT is limited to **imitation** — the student only learns what the teacher showed it.  
What if the student could **explore** and learn from its own mistakes? → That's RFT (next notebook).

---

## 4. Deploy & Test the Fine-Tuned Agent

Same agent code, just swap the model deployment:

```bash
# Deploy the SFT-finetuned agent
cd deploy && azd deploy retail-gpt-4-1-finetuned

# Invoke it on a tricky scenario
azd ai agent invoke --agent-name retail-gpt-4-1-finetuned \
  --message "I want to exchange the yoga mat from ORD-007 for a different color"
```

**Compare:** The base GPT-4.1 might get this wrong (deny instead of exchange), but the fine-tuned version handles it correctly — it learned the exchange flow from the teacher.

### 🔗 Resources:

| Resource | Link |
|----------|------|
| Fine-tuned agent source | [`deploy/src/retail-gpt-4-1-finetuned/`](../deploy/src/retail-gpt-4-1-finetuned/) |
| Mini fine-tuned agent | [`deploy/src/retail-gpt-4-1-mini-finetuned/`](../deploy/src/retail-gpt-4-1-mini-finetuned/) |
| Nano fine-tuned agent | [`deploy/src/retail-gpt-4-1-nano-finetuned/`](../deploy/src/retail-gpt-4-1-nano-finetuned/) |
| Azure AI Foundry | [Fine-tuning Jobs](https://ai.azure.com) |

---

## 📊 Summary

| What | How |
|------|-----|
| **Method** | Supervised Fine-Tuning (imitation learning) |
| **Data** | 481 teacher traces from GPT-5.4 |
| **Effort** | One API call per model |
| **Result** | Students beat teacher by 13-41 points |
| **Limitation** | Can only imitate — can't discover new strategies |

---

**Next → [`3_rft_foundry.ipynb`](./3_rft_foundry.ipynb)** — Reinforcement Fine-Tuning with the Foundry SDK.  
*Let o4-mini learn from trial-and-error with a grader as reward signal — going beyond imitation.*